# Pneumonia detection from rontgen pictures(CNN)

Chest rontgen pictures → `NORMAL` / `PNEUMONIA`

In [ ]:
import os

DATA_DIR = "data/chest_xray"

for split in ["train", "test"]:
    for kelas in ["NORMAL", "PNEUMONIA"]:
        print(split, kelas, len(os.listdir(f"{DATA_DIR}/{split}/{kelas}")))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, classification_report

#activate gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

#data augmentation 
tf = transforms.Compose([
    transforms.Grayscale(), 
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

#split data
train_data = ImageFolder(f"{DATA_DIR}/train", transform=tf)
test_data = ImageFolder(f"{DATA_DIR}/test", transform=tf)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

# NORMAL lebih sedikit, jadi bobotnya lebih besar
bobot = torch.tensor([3.0, 1.0]).to(device)

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),   # 128 → 64
    
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(), 
    nn.MaxPool2d(2),  # 64 → 32
    
    nn.Flatten(),
    nn.Linear(32 * 32 * 32, 64), nn.ReLU(),
    nn.Linear(64, 2),
).to(device)

loss_fn = nn.CrossEntropyLoss(weight=bobot)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# training
for epoch in range(5):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss {total_loss / len(train_loader):.4f}")

# evaluasi
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        y_true += y.tolist()
        y_pred += pred.tolist()

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=train_data.classes))